# Descriptive Statistics — Flood AI Dataset

This notebook computes descriptive statistics (mean, min, max, standard deviation) for all features across both models (Model 1 and Model 2), covering:
- **1D static node features** (pipe/channel network)
- **2D static node features** (surface mesh)
- **1D dynamic features** — water level across all events
- **2D dynamic features** — water level across all events
- **Rainfall** — per-event rainfall signal

Statistics are computed on **raw (un-normalised) values** across all events in the dataset.

In [1]:
# configuration

# Platform: 'kaggle' or 'colab'
PLATFORM = 'kaggle'  # <- EDIT

# google colba only
DRIVE_DATA_PATH = '/content/drive/MyDrive/ClimateProject/Models'  # <- EDIT

MODEL_IDS = [1, 2]

In [2]:
# imports and paths

import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)

if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path(DRIVE_DATA_PATH)
else:
    _hits = list(Path('/kaggle/input').rglob('1d_nodes_static.csv'))
    if _hits:
        _d = _hits[0].parent
        if _d.name in ('train', 'test'):
            _d = _d.parent
        BASE = _d.parent
    else:
        BASE = Path('/kaggle/input')

print(f'BASE: {BASE}')
for mid in MODEL_IDS:
    mp = BASE / f'Model_{mid}'
    print(f'  Model_{mid} exists: {mp.exists()}')

BASE: /kaggle/input/datasets/timweischueler/zip-data/Models
  Model_1 exists: True
  Model_2 exists: True


In [3]:
# helper functions

def find_static(model_path, name):
    """find a static CSV under model_path (checks train/, test/, root)"""
    for sub in ['train', 'test']:
        p = Path(model_path) / sub / name
        if p.exists():
            return p
    p = Path(model_path) / name
    if p.exists():
        return p
    hits = list(Path(model_path).rglob(name))
    if hits:
        return hits[0]
    raise FileNotFoundError(f'{name} not found under {model_path}')


def get_event_dirs(model_path):
    """sorted list of event dirs under model_path/train/"""
    train_dir = Path(model_path) / 'train'
    if not train_dir.exists():
        hits = [d for d in Path(model_path).rglob('train') if d.is_dir()]
        train_dir = hits[0] if hits else train_dir
    return sorted(
        [d for d in train_dir.iterdir() if d.is_dir() and d.name.startswith('event_')],
        key=lambda d: int(d.name.split('_')[1])
    )


def load_static(model_path):
    """load static node feature CSVs for 1D and 2D"""
    cols_1d = ['depth', 'invert_elevation', 'surface_elevation', 'base_area']
    cols_2d = ['area', 'roughness', 'min_elevation', 'elevation',
               'aspect', 'curvature', 'flow_accumulation']

    n1d = pd.read_csv(find_static(model_path, '1d_nodes_static.csv'))
    n2d = pd.read_csv(find_static(model_path, '2d_nodes_static.csv'))

    cols_1d_present = [c for c in cols_1d if c in n1d.columns]
    cols_2d_present = [c for c in cols_2d if c in n2d.columns]

    return n1d[cols_1d_present], n2d[cols_2d_present]


def load_all_dynamic(model_path):
    """load dynamic data across all events, returns raw arrays"""
    event_dirs = get_event_dirs(model_path)
    wl1d_all, wl2d_all, rain_all = [], [], []

    for ev in event_dirs:
        try:
            d1 = pd.read_csv(ev / '1d_nodes_dynamic_all.csv')
            d2 = pd.read_csv(ev / '2d_nodes_dynamic_all.csv')

            wl1d = d1.pivot(
                index='timestep', columns='node_idx', values='water_level'
            ).ffill().bfill().values.astype('float32')

            wl2d = d2.pivot(
                index='timestep', columns='node_idx', values='water_level'
            ).ffill().bfill().values.astype('float32')

            rain = d2.groupby('timestep')['rainfall'].mean().fillna(0).values.astype('float32')

            wl1d_all.append(wl1d)
            wl2d_all.append(wl2d)
            rain_all.append(rain)
        except Exception as e:
            print(f'  Skipping {ev.name}: {e}')

    return wl1d_all, wl2d_all, rain_all, len(event_dirs)


def stats_table(arr_flat, feature_name):
    """descriptive stats for a flat array, returns one-row DataFrame"""
    arr = arr_flat[~np.isnan(arr_flat)]
    return pd.DataFrame([{
        'Feature':  feature_name,
        'Mean':     float(np.mean(arr)),
        'Std':      float(np.std(arr)),
        'Min':      float(np.min(arr)),
        'Max':      float(np.max(arr)),
        'N (values)': len(arr)
    }])


def static_stats(df):
    """per-column descriptive stats for a static feature DataFrame"""
    rows = []
    for col in df.columns:
        arr = df[col].dropna().values
        rows.append({
            'Feature':    col,
            'Mean':       float(np.mean(arr)),
            'Std':        float(np.std(arr)),
            'Min':        float(np.min(arr)),
            'Max':        float(np.max(arr)),
            'N (nodes)':  len(arr)
        })
    return pd.DataFrame(rows).set_index('Feature')


print('Helper functions defined.')

Helper functions defined.


In [4]:
# load all data

data = {}

for mid in MODEL_IDS:
    mp = BASE / f'Model_{mid}'
    print(f'\n=== Model {mid} ===')

    n1d, n2d = load_static(mp)
    print(f'  1D static: {len(n1d)} nodes, columns: {list(n1d.columns)}')
    print(f'  2D static: {len(n2d)} nodes, columns: {list(n2d.columns)}')

    wl1d_all, wl2d_all, rain_all, n_events = load_all_dynamic(mp)
    print(f'  Events loaded: {len(wl1d_all)} / {n_events}')
    if wl1d_all:
        total_ts = sum(w.shape[0] for w in wl1d_all)
        print(f'  Total 1D timesteps (across all events): {total_ts}')
        print(f'  Total 2D timesteps (across all events): {sum(w.shape[0] for w in wl2d_all)}')

    data[mid] = {
        'n1d': n1d, 'n2d': n2d,
        'wl1d': wl1d_all, 'wl2d': wl2d_all, 'rain': rain_all,
        'n_events': len(wl1d_all)
    }

print('\nData loading complete.')


=== Model 1 ===
  1D static: 17 nodes, columns: ['depth', 'invert_elevation', 'surface_elevation', 'base_area']
  2D static: 3716 nodes, columns: ['area', 'roughness', 'min_elevation', 'elevation', 'aspect', 'curvature', 'flow_accumulation']
  Events loaded: 68 / 68
  Total 1D timesteps (across all events): 15707
  Total 2D timesteps (across all events): 15707

=== Model 2 ===
  1D static: 198 nodes, columns: ['depth', 'invert_elevation', 'surface_elevation', 'base_area']
  2D static: 4299 nodes, columns: ['area', 'roughness', 'min_elevation', 'elevation', 'aspect', 'curvature', 'flow_accumulation']
  Events loaded: 69 / 69
  Total 1D timesteps (across all events): 17853
  Total 2D timesteps (across all events): 17853

Data loading complete.


In [5]:
# 1D static node features

print('=' * 70)
print('1D STATIC NODE FEATURES (pipe/channel network)')
print('Statistics computed across all nodes. Raw (un-normalised) values.')
print('=' * 70)

for mid in MODEL_IDS:
    tbl = static_stats(data[mid]['n1d'])
    print(f'\n--- Model {mid}  ({len(data[mid]["n1d"])} nodes) ---')
    display(tbl.style
              .format('{:.4f}', subset=['Mean', 'Std', 'Min', 'Max'])
              .format('{:,.0f}', subset=['N (nodes)'])
              .set_caption(f'Model {mid} — 1D Static Node Features')
              .set_table_styles([{'selector': 'caption',
                                  'props': [('font-size', '13px'),
                                            ('font-weight', 'bold')]}]))

1D STATIC NODE FEATURES (pipe/channel network)
Statistics computed across all nodes. Raw (un-normalised) values.

--- Model 1  (17 nodes) ---


,Mean,Std,Min,Max,N (nodes)
Feature,,,,,
depth,4.4026,2.5637,0.4700,9.9020,17
invert_elevation,308.0298,17.0171,286.6000,348.4260,17
surface_elevation,312.4324,17.2248,287.0700,350.8500,17
base_area,14.0376,9.5191,0.0000,50.2400,17



--- Model 2  (198 nodes) ---


,Mean,Std,Min,Max,N (nodes)
Feature,,,,,
depth,6.1719,3.3489,-1.0000,25.0400,198
invert_elevation,35.5543,3.7962,23.0000,44.5900,198
surface_elevation,41.7262,3.3609,22.0000,51.7100,198
base_area,24.6723,17.7702,0.0000,108.0000,198


In [6]:
# 2D static node features

print('=' * 70)
print('2D STATIC NODE FEATURES (surface mesh)')
print('Statistics computed across all nodes. Raw (un-normalised) values.')
print('=' * 70)

for mid in MODEL_IDS:
    tbl = static_stats(data[mid]['n2d'])
    print(f'\n--- Model {mid}  ({len(data[mid]["n2d"])} nodes) ---')
    display(tbl.style
              .format('{:.4f}', subset=['Mean', 'Std', 'Min', 'Max'])
              .format('{:,.0f}', subset=['N (nodes)'])
              .set_caption(f'Model {mid} — 2D Static Node Features')
              .set_table_styles([{'selector': 'caption',
                                  'props': [('font-size', '13px'),
                                            ('font-weight', 'bold')]}]))

2D STATIC NODE FEATURES (surface mesh)
Statistics computed across all nodes. Raw (un-normalised) values.

--- Model 1  (3716 nodes) ---


,Mean,Std,Min,Max,N (nodes)
Feature,,,,,
area,609.3919,116.1106,-0.0000,1218.5624,"3,716"
roughness,0.0586,0.0186,0.0130,0.1000,"3,716"
min_elevation,321.9399,14.4443,293.1492,359.8446,"3,704"
elevation,323.1608,14.4956,293.8125,360.2188,"3,716"
aspect,184.8906,115.0799,-1.0000,359.9293,"3,716"
curvature,0.0001,0.0003,0.0000,0.0052,"3,716"
flow_accumulation,1.5137,1.4074,1.0000,31.0000,"3,716"



--- Model 2  (4299 nodes) ---


,Mean,Std,Min,Max,N (nodes)
Feature,,,,,
area,15125.8026,8464.7714,2079.4333,53465.6450,"4,299"
roughness,0.0600,0.0000,0.0600,0.0600,"4,299"
min_elevation,42.8932,2.9970,32.8848,54.7352,"4,299"
elevation,44.1707,3.1327,34.5625,60.4688,"4,299"
aspect,171.1045,107.6852,-1.0000,359.9928,"4,299"
curvature,0.0000,0.0001,0.0000,0.0047,"4,299"
flow_accumulation,1.7411,1.7084,1.0000,21.0000,"4,299"


In [7]:
# 1D dynamic: water level

print('=' * 70)
print('1D DYNAMIC FEATURE: Water Level')
print('Statistics aggregated across all events × timesteps × nodes.')
print('=' * 70)

rows = []
for mid in MODEL_IDS:
    wl_all = np.concatenate([w.flatten() for w in data[mid]['wl1d']])
    n_events  = data[mid]['n_events']
    total_ts  = sum(w.shape[0] for w in data[mid]['wl1d'])
    n_nodes   = data[mid]['wl1d'][0].shape[1] if data[mid]['wl1d'] else 0
    rows.append({
        'Model':       f'Model {mid}',
        'Events':      n_events,
        'Nodes (1D)':  n_nodes,
        'Total timesteps': total_ts,
        'Mean [m]':    float(np.mean(wl_all)),
        'Std [m]':     float(np.std(wl_all)),
        'Min [m]':     float(np.min(wl_all)),
        'Max [m]':     float(np.max(wl_all)),
    })

tbl = pd.DataFrame(rows).set_index('Model')
display(tbl.style
          .format('{:.4f}', subset=['Mean [m]', 'Std [m]', 'Min [m]', 'Max [m]'])
          .format('{:,.0f}', subset=['Events', 'Nodes (1D)', 'Total timesteps'])
          .set_caption('1D Water Level — Aggregated across all events')
          .set_table_styles([{'selector': 'caption',
                              'props': [('font-size', '13px'), ('font-weight', 'bold')]}]))

# Per-event detail
print('\n--- Per-event summary (1D Water Level) ---')
for mid in MODEL_IDS:
    ev_rows = []
    for i, w in enumerate(data[mid]['wl1d']):
        flat = w.flatten()
        ev_rows.append({'Event': i, 'Timesteps': w.shape[0],
                        'Mean [m]': float(np.mean(flat)), 'Std [m]': float(np.std(flat)),
                        'Min [m]': float(np.min(flat)),  'Max [m]': float(np.max(flat))})
    ev_tbl = pd.DataFrame(ev_rows).set_index('Event')
    print(f'\nModel {mid}')
    display(ev_tbl.style
              .format('{:.4f}', subset=['Mean [m]', 'Std [m]', 'Min [m]', 'Max [m]'])
              .format('{:,.0f}', subset=['Timesteps'])
              .set_caption(f'Model {mid} — 1D Water Level per Event')
              .set_table_styles([{'selector': 'caption',
                                  'props': [('font-size', '12px'), ('font-weight', 'bold')]}]))

1D DYNAMIC FEATURE: Water Level
Statistics aggregated across all events × timesteps × nodes.


,Events,Nodes (1D),Total timesteps,Mean [m],Std [m],Min [m],Max [m]
Model,,,,,,,
Model 1,68,17,"15,707",308.0884,16.8777,286.6000,347.9525
Model 2,69,198,"17,853",39.8941,3.1918,23.0000,48.0672



--- Per-event summary (1D Water Level) ---

Model 1


,Timesteps,Mean [m],Std [m],Min [m],Max [m]
Event,,,,,
0,94,308.2726,16.7953,286.6735,347.9525
1,205,308.0894,16.8746,286.6634,347.9525
2,205,308.1827,16.8276,286.6000,347.9525
3,445,308.0412,16.8977,286.6000,347.9525
4,97,308.1500,16.8522,286.6702,347.9525
5,205,308.1214,16.8595,286.6630,347.9525
6,97,308.1918,16.8370,286.6708,347.9525
7,205,308.2222,16.8138,286.6637,347.9525
8,445,308.0411,16.8978,286.6000,347.9525



Model 2


,Timesteps,Mean [m],Std [m],Min [m],Max [m]
Event,,,,,
0,97,39.8885,2.4866,28.8535,47.0472
1,205,40.6393,2.1599,31.9939,47.3187
2,205,39.8295,2.4074,28.4728,46.3721
3,205,39.9496,2.5088,25.3724,46.8136
4,97,38.7221,2.9668,27.2999,46.1015
5,205,40.8740,2.2250,28.2578,46.7177
6,97,40.7941,2.1183,30.8961,47.0845
7,205,41.4088,2.2540,26.6998,47.4429
8,205,40.4543,2.7107,25.0244,46.9787


In [8]:
# 2D dynamic: water level

print('=' * 70)
print('2D DYNAMIC FEATURE: Water Level')
print('Statistics aggregated across all events × timesteps × nodes.')
print('=' * 70)

rows = []
for mid in MODEL_IDS:
    wl_all = np.concatenate([w.flatten() for w in data[mid]['wl2d']])
    n_events  = data[mid]['n_events']
    total_ts  = sum(w.shape[0] for w in data[mid]['wl2d'])
    n_nodes   = data[mid]['wl2d'][0].shape[1] if data[mid]['wl2d'] else 0
    rows.append({
        'Model':       f'Model {mid}',
        'Events':      n_events,
        'Nodes (2D)':  n_nodes,
        'Total timesteps': total_ts,
        'Mean [m]':    float(np.mean(wl_all)),
        'Std [m]':     float(np.std(wl_all)),
        'Min [m]':     float(np.min(wl_all)),
        'Max [m]':     float(np.max(wl_all)),
    })

tbl = pd.DataFrame(rows).set_index('Model')
display(tbl.style
          .format('{:.4f}', subset=['Mean [m]', 'Std [m]', 'Min [m]', 'Max [m]'])
          .format('{:,.0f}', subset=['Events', 'Nodes (2D)', 'Total timesteps'])
          .set_caption('2D Water Level — Aggregated across all events')
          .set_table_styles([{'selector': 'caption',
                              'props': [('font-size', '13px'), ('font-weight', 'bold')]}]))

# Per-event detail
print('\n--- Per-event summary (2D Water Level) ---')
for mid in MODEL_IDS:
    ev_rows = []
    for i, w in enumerate(data[mid]['wl2d']):
        flat = w.flatten()
        ev_rows.append({'Event': i, 'Timesteps': w.shape[0],
                        'Mean [m]': float(np.mean(flat)), 'Std [m]': float(np.std(flat)),
                        'Min [m]': float(np.min(flat)),  'Max [m]': float(np.max(flat))})
    ev_tbl = pd.DataFrame(ev_rows).set_index('Event')
    print(f'\nModel {mid}')
    display(ev_tbl.style
              .format('{:.4f}', subset=['Mean [m]', 'Std [m]', 'Min [m]', 'Max [m]'])
              .format('{:,.0f}', subset=['Timesteps'])
              .set_caption(f'Model {mid} — 2D Water Level per Event')
              .set_table_styles([{'selector': 'caption',
                                  'props': [('font-size', '12px'), ('font-weight', 'bold')]}]))

2D DYNAMIC FEATURE: Water Level
Statistics aggregated across all events × timesteps × nodes.


,Events,Nodes (2D),Total timesteps,Mean [m],Std [m],Min [m],Max [m]
Model,,,,,,,
Model 1,68,"3,716","15,707",322.1122,14.3788,293.1492,360.0810
Model 2,69,"4,299","17,853",43.7300,2.7271,32.8848,55.3354



--- Per-event summary (2D Water Level) ---

Model 1


,Timesteps,Mean [m],Std [m],Min [m],Max [m]
Event,,,,,
0,94,322.1090,14.3620,293.2058,360.0246
1,205,322.1089,14.3941,293.2055,359.9798
2,205,322.1342,14.4122,293.1492,359.9719
3,445,322.1315,14.3577,293.1492,359.9649
4,97,322.0811,14.3816,293.2057,360.0323
5,205,322.1420,14.3663,293.2055,359.9901
6,97,322.0836,14.3837,293.2057,360.0356
7,205,322.1771,14.3425,293.2055,360.0262
8,445,322.1321,14.3567,293.1492,359.9699



Model 2


,Timesteps,Mean [m],Std [m],Min [m],Max [m]
Event,,,,,
0,97,43.6032,2.8127,33.1015,55.2643
1,205,43.7531,2.6902,36.3682,55.3354
2,205,43.6002,2.8411,33.2159,55.1428
3,205,43.6383,2.7889,33.1821,55.2032
4,97,43.4567,2.9064,33.1011,55.2182
5,205,43.8064,2.6986,33.2503,55.2040
6,97,43.7623,2.6864,35.7562,55.2431
7,205,43.9986,2.5523,33.2231,55.2699
8,205,43.7822,2.7210,33.1864,55.2212


In [9]:
# rainfall stats

print('=' * 70)
print('RAINFALL (mean over 2D nodes per timestep)')
print('Statistics aggregated across all events × timesteps.')
print('=' * 70)

rows = []
for mid in MODEL_IDS:
    rain_all = np.concatenate(data[mid]['rain'])
    rain_nonzero = rain_all[rain_all > 0]
    n_events  = data[mid]['n_events']
    total_ts  = len(rain_all)
    rows.append({
        'Model':              f'Model {mid}',
        'Events':             n_events,
        'Total timesteps':    total_ts,
        'Mean [mm/h]':        float(np.mean(rain_all)),
        'Std [mm/h]':         float(np.std(rain_all)),
        'Min [mm/h]':         float(np.min(rain_all)),
        'Max [mm/h]':         float(np.max(rain_all)),
        'Mean (wet) [mm/h]':  float(np.mean(rain_nonzero)) if len(rain_nonzero) > 0 else 0.0,
        '% wet timesteps':    100.0 * len(rain_nonzero) / total_ts if total_ts > 0 else 0.0,
    })

tbl = pd.DataFrame(rows).set_index('Model')
display(tbl.style
          .format('{:.4f}', subset=['Mean [mm/h]', 'Std [mm/h]', 'Min [mm/h]',
                                    'Max [mm/h]', 'Mean (wet) [mm/h]'])
          .format('{:.1f}%', subset=['% wet timesteps'])
          .format('{:,.0f}', subset=['Events', 'Total timesteps'])
          .set_caption('Rainfall — Aggregated across all events')
          .set_table_styles([{'selector': 'caption',
                              'props': [('font-size', '13px'), ('font-weight', 'bold')]}]))

# Per-event detail
print('\n--- Per-event rainfall summary ---')
for mid in MODEL_IDS:
    ev_rows = []
    for i, r in enumerate(data[mid]['rain']):
        rn = r[r > 0]
        ev_rows.append({
            'Event':             i,
            'Timesteps':         len(r),
            'Mean [mm/h]':       float(np.mean(r)),
            'Std [mm/h]':        float(np.std(r)),
            'Max [mm/h]':        float(np.max(r)),
            'Mean (wet) [mm/h]': float(np.mean(rn)) if len(rn) > 0 else 0.0,
            '% wet':             100.0 * len(rn) / len(r) if len(r) > 0 else 0.0,
        })
    ev_tbl = pd.DataFrame(ev_rows).set_index('Event')
    print(f'\nModel {mid}')
    display(ev_tbl.style
              .format('{:.4f}', subset=['Mean [mm/h]', 'Std [mm/h]', 'Max [mm/h]', 'Mean (wet) [mm/h]'])
              .format('{:.1f}%', subset=['% wet'])
              .format('{:,.0f}', subset=['Timesteps'])
              .set_caption(f'Model {mid} — Rainfall per Event')
              .set_table_styles([{'selector': 'caption',
                                  'props': [('font-size', '12px'), ('font-weight', 'bold')]}]))

RAINFALL (mean over 2D nodes per timestep)
Statistics aggregated across all events × timesteps.


,Events,Total timesteps,Mean [mm/h],Std [mm/h],Min [mm/h],Max [mm/h],Mean (wet) [mm/h],% wet timesteps
Model,,,,,,,,
Model 1,68,"15,707",0.0188,0.0285,0.0000,0.3821,0.0310,60.4%
Model 2,69,"17,853",0.0169,0.0300,0.0000,0.2933,0.0310,54.6%



--- Per-event rainfall summary ---

Model 1


,Timesteps,Mean [mm/h],Std [mm/h],Max [mm/h],Mean (wet) [mm/h],% wet
Event,,,,,,
0,94,0.0287,0.0530,0.1562,0.1225,23.4%
1,205,0.0187,0.0178,0.0598,0.0302,62.0%
2,205,0.0259,0.0140,0.0490,0.0277,93.7%
3,445,0.0148,0.0133,0.0407,0.0228,64.7%
4,97,0.0220,0.0448,0.1794,0.0970,22.7%
5,205,0.0216,0.0204,0.0764,0.0349,62.0%
6,97,0.0266,0.0519,0.1894,0.1173,22.7%
7,205,0.0302,0.0352,0.1595,0.0487,62.0%
8,445,0.0147,0.0139,0.0465,0.0228,64.7%



Model 2


,Timesteps,Mean [mm/h],Std [mm/h],Max [mm/h],Mean (wet) [mm/h],% wet
Event,,,,,,
0,97,0.0169,0.0396,0.1567,0.0781,21.6%
1,205,0.0152,0.0511,0.2933,0.0511,29.8%
2,205,0.0134,0.0123,0.0350,0.0217,62.0%
3,205,0.0152,0.0202,0.0750,0.0246,62.0%
4,97,0.0076,0.0170,0.0933,0.0333,22.7%
5,205,0.0212,0.0228,0.0767,0.0343,62.0%
6,97,0.0138,0.0302,0.1233,0.0608,22.7%
7,205,0.0299,0.0481,0.1583,0.0631,47.3%
8,205,0.0233,0.0266,0.0933,0.0376,62.0%


In [10]:
# cross-model comparison summary

print('=' * 70)
print('CROSS-MODEL COMPARISON SUMMARY')
print('=' * 70)

summary_rows = []
for mid in MODEL_IDS:
    wl1d = np.concatenate([w.flatten() for w in data[mid]['wl1d']])
    wl2d = np.concatenate([w.flatten() for w in data[mid]['wl2d']])
    rain = np.concatenate(data[mid]['rain'])

    summary_rows.append({
        'Model':                   f'Model {mid}',
        '# Events':                data[mid]['n_events'],
        '1D nodes':                data[mid]['n1d'].shape[0],
        '2D nodes':                data[mid]['n2d'].shape[0],
        'WL-1D mean [m]':          float(np.mean(wl1d)),
        'WL-1D std [m]':           float(np.std(wl1d)),
        'WL-1D range [m]':         float(np.max(wl1d) - np.min(wl1d)),
        'WL-2D mean [m]':          float(np.mean(wl2d)),
        'WL-2D std [m]':           float(np.std(wl2d)),
        'WL-2D range [m]':         float(np.max(wl2d) - np.min(wl2d)),
        'Rain mean [mm/h]':        float(np.mean(rain)),
        'Rain max [mm/h]':         float(np.max(rain)),
    })

summary_tbl = pd.DataFrame(summary_rows).set_index('Model').T
display(summary_tbl.style
          .set_caption('Cross-Model Comparison Summary')
          .set_table_styles([{'selector': 'caption',
                              'props': [('font-size', '14px'), ('font-weight', 'bold')]}]))

print('\nDescriptive statistics complete.')

CROSS-MODEL COMPARISON SUMMARY


Model,Model 1,Model 2
# Events,68.000000,69.000000
1D nodes,17.000000,198.000000
2D nodes,3716.000000,4299.000000
WL-1D mean [m],308.088379,39.894089
WL-1D std [m],16.877748,3.191783
WL-1D range [m],61.352448,25.067162
WL-2D mean [m],322.112213,43.729977
WL-2D std [m],14.378805,2.727131
WL-2D range [m],66.931824,22.450623
Rain mean [mm/h],0.018751,0.016932



Descriptive statistics complete.
